# Análisis Exploratorio de Datos
## Predicción de Ingresos Diarios de Cafeterías
**UPTC — Ingeniería de Sistemas — Machine Learning 2026¨**

Este notebook sigue el **ciclo de vida estándar de ML** y documenta los pasos 1 al 4:
1. Recopilación de datos
2. Elección de medida de éxito
3. Protocolo de evaluación
4. Preparación de los datos

##  Instalación de dependencias
Ejecuta esta celda **una sola vez** si es la primera vez que corres el notebook.

In [ ]:
import subprocess, sys
subprocess.run([
    sys.executable, "-m", "pip", "install",
    "pandas", "numpy", "matplotlib", "seaborn",
    "scikit-learn", "scipy", "joblib", "--quiet"
])
print("✅ Dependencias instaladas correctamente")

## Paso 0 — Importaciones y configuración

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

# Configuración visual
sns.set_theme(style="whitegrid", palette="muted")
plt.rcParams["figure.figsize"] = (11, 5)
plt.rcParams["figure.dpi"]    = 120
pd.set_option("display.float_format", "{:.4f}".format)

# ── Rutas ─────────────────────────────────────────────────────────────
DATA_PATH   = "../data/coffee_shop_revenue.csv"
GRAPHS_PATH = "../results/graphs/"
TARGET      = "daily_revenue"   # columna objetivo real del CSV
RANDOM_STATE = 42

print("✅ Librerías cargadas")

---
##  Paso 1 — Recopilación de datos
**Fuente:** Kaggle — *Coffee Shop Daily Revenue Prediction Dataset*  
**Variable objetivo:** `daily_revenue` (valor continuo en USD)

In [ ]:
df = pd.read_csv(DATA_PATH)

print(f" Dimensiones del dataset: {df.shape[0]} filas × {df.shape[1]} columnas")
print(f"\n Columnas disponibles:\n{list(df.columns)}")
df.head()

In [ ]:
print("\n Tipos de datos y valores no nulos:")
df.info()

print("\n  Valores nulos por columna:")
nulos = df.isnull().sum()
print(nulos[nulos >= 0].to_string())

print(f"\n Filas duplicadas: {df.duplicated().sum()}")

In [ ]:
print(" Estadísticas descriptivas:")
df.describe()

---
##  Paso 2 — Medida de éxito

La variable objetivo `daily_revenue` es **continua** → problema de **regresión**.

| Métrica | Descripción | Cuándo usarla |
|---------|-------------|---------------|
| **MAE** | Error absoluto medio en USD | Interpretación directa |
| **RMSE** | Raíz del error cuadrático medio | Penaliza errores grandes |
| **R²** | Proporción de varianza explicada (0–1) | Comparar modelos |
| **Accuracy / F1** | Para Regresión Logística (binaria) | Solo clasificación |

> **Nota:** La Regresión Logística trabaja con una versión binarizada del ingreso (alto/bajo según la mediana del conjunto de entrenamiento).

---
##  Paso 3 — Protocolo de evaluación

- **División:** 70% entrenamiento / 15% validación / 15% prueba
- **Semilla:** `random_state = 42` en todos los modelos
- **Normalización:** `StandardScaler` ajustado **solo** sobre train (sin fuga de datos)
- **Evaluación final:** todos los modelos comparados sobre el **mismo conjunto de test**

In [ ]:
fig, ax = plt.subplots(figsize=(5, 5))
labels  = ["Train (70%)", "Validación (15%)", "Test (15%)"]
sizes   = [70, 15, 15]
colors  = ["#4C72B0", "#55A868", "#DD8452"]
wedges, texts, autotexts = ax.pie(
    sizes, labels=labels, autopct="%1.0f%%",
    colors=colors, startangle=90,
    wedgeprops=dict(edgecolor="white", linewidth=2)
)
for t in autotexts:
    t.set_fontsize(12)
    t.set_fontweight("bold")
ax.set_title("Protocolo de División del Dataset", fontsize=13, fontweight="bold", pad=15)
plt.tight_layout()
plt.savefig(f"{GRAPHS_PATH}protocolo_division.png", dpi=150, bbox_inches="tight")
plt.show()
print(" Gráfica guardada")

---
##  Paso 4 — Preparación de los datos
### 4.1 Análisis de la variable objetivo

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
fig.suptitle(f"Distribución de '{TARGET}'", fontsize=14, fontweight="bold")

# Histograma
axes[0].hist(df[TARGET], bins=40, color="#4C72B0", edgecolor="white")
axes[0].set_xlabel("daily_revenue (USD)")
axes[0].set_ylabel("Frecuencia")
axes[0].set_title("Histograma")

# Boxplot
axes[1].boxplot(df[TARGET], vert=True, patch_artist=True,
                boxprops=dict(facecolor="#4C72B0", alpha=0.7),
                medianprops=dict(color="white", linewidth=2.5))
axes[1].set_ylabel("daily_revenue (USD)")
axes[1].set_title("Boxplot")

# QQ Plot
stats.probplot(df[TARGET], dist="norm", plot=axes[2])
axes[2].set_title("QQ Plot (¿Normal?)")
axes[2].get_lines()[0].set(color="#4C72B0", alpha=0.5, markersize=3)
axes[2].get_lines()[1].set(color="#DD8452", linewidth=2)

plt.tight_layout()
plt.savefig(f"{GRAPHS_PATH}01_distribucion_objetivo.png", dpi=150, bbox_inches="tight")
plt.show()

print(f"Media    : ${df[TARGET].mean():,.2f}")
print(f"Mediana  : ${df[TARGET].median():,.2f}")
print(f"Std      : ${df[TARGET].std():,.2f}")
print(f"Sesgo    : {df[TARGET].skew():.4f}")
print(f"Curtosis : {df[TARGET].kurt():.4f}")

### 4.2 Distribución de todas las variables numéricas

In [ ]:
num_cols = df.select_dtypes(include=[np.number]).columns.tolist()
n = len(num_cols)
cols_grid = 3
rows_grid = (n + cols_grid - 1) // cols_grid

fig, axes = plt.subplots(rows_grid, cols_grid, figsize=(14, rows_grid * 3.5))
axes = axes.flatten()

for i, col in enumerate(num_cols):
    axes[i].hist(df[col].dropna(), bins=30, color="#4C72B0", edgecolor="white", alpha=0.8)
    axes[i].set_title(col, fontsize=10)
    axes[i].set_ylabel("Frecuencia")

for j in range(i + 1, len(axes)):
    axes[j].set_visible(False)

fig.suptitle("Distribución de Variables Numéricas", fontsize=13, fontweight="bold", y=1.01)
plt.tight_layout()
plt.show()

### 4.3 Mapa de correlación

In [ ]:
corr = df.select_dtypes(include=[np.number]).corr()
mask = np.triu(np.ones_like(corr, dtype=bool))

plt.figure(figsize=(10, 7))
sns.heatmap(
    corr, mask=mask, annot=True, fmt=".2f",
    cmap="coolwarm", center=0, vmin=-1, vmax=1,
    linewidths=0.5, linecolor="white",
    cbar_kws={"shrink": 0.8}
)
plt.title("Mapa de Correlación — Variables Numéricas", fontsize=13, fontweight="bold")
plt.tight_layout()
plt.savefig(f"{GRAPHS_PATH}02_correlacion.png", dpi=150, bbox_inches="tight")
plt.show()

# Variables más correlacionadas con el target
print(f"\nCorrelación con '{TARGET}' (ordenada):")
print(corr[TARGET].drop(TARGET).sort_values(ascending=False).to_string())

### 4.4 Variables categóricas vs Revenue

In [ ]:
cat_cols = df.select_dtypes(include=["object", "category"]).columns.tolist()

if cat_cols:
    fig, axes = plt.subplots(1, len(cat_cols), figsize=(6 * len(cat_cols), 5))
    if len(cat_cols) == 1:
        axes = [axes]
    for ax, col in zip(axes, cat_cols):
        order = df.groupby(col)[TARGET].median().sort_values(ascending=False).index
        sns.boxplot(data=df, x=col, y=TARGET, order=order, ax=ax,
                    palette="muted", width=0.5)
        ax.set_title(f"{col} vs {TARGET}", fontsize=11, fontweight="bold")
        ax.set_xlabel(col)
        ax.set_ylabel("daily_revenue (USD)")
        ax.tick_params(axis="x", rotation=30)
    plt.suptitle("Variables Categóricas vs Ingreso Diario", fontsize=13, fontweight="bold", y=1.02)
    plt.tight_layout()
    plt.show()
else:
    print("No hay variables categóricas en el dataset.")

### 4.5 Detección de outliers (método IQR)

In [ ]:
Q1  = df[TARGET].quantile(0.25)
Q3  = df[TARGET].quantile(0.75)
IQR = Q3 - Q1
lower = Q1 - 1.5 * IQR
upper = Q3 + 1.5 * IQR

outliers = df[(df[TARGET] < lower) | (df[TARGET] > upper)]

print(f"Q1      : ${Q1:,.2f}")
print(f"Q3      : ${Q3:,.2f}")
print(f"IQR     : ${IQR:,.2f}")
print(f"Límite inferior : ${lower:,.2f}")
print(f"Límite superior : ${upper:,.2f}")
print(f"\n Outliers detectados: {len(outliers)} ({len(outliers)/len(df)*100:.1f}% del total)")

fig, ax = plt.subplots(figsize=(9, 3))
ax.boxplot(df[TARGET], vert=False, patch_artist=True,
           boxprops=dict(facecolor="#4C72B0", alpha=0.6),
           medianprops=dict(color="white", linewidth=2),
           flierprops=dict(marker="o", color="#DD8452", markersize=4))
ax.set_xlabel("daily_revenue (USD)")
ax.set_title("Outliers en daily_revenue", fontsize=12, fontweight="bold")
plt.tight_layout()
plt.show()

### 4.6 Limpieza y transformaciones

In [ ]:
# ── Duplicados ────────────────────────────────────────────────────────
n_before = len(df)
df_clean = df.drop_duplicates()
print(f"Duplicados eliminados : {n_before - len(df_clean)}")

# ── Imputación de nulos (mediana) ─────────────────────────────────────
num_cols_clean = df_clean.select_dtypes(include=[np.number]).columns
for col in num_cols_clean:
    if df_clean[col].isnull().sum() > 0:
        df_clean[col] = df_clean[col].fillna(df_clean[col].median())
        print(f"  → '{col}' imputado con mediana")

# ── One-hot encoding ──────────────────────────────────────────────────
cat_cols_clean = df_clean.select_dtypes(include=["object", "category"]).columns.tolist()
df_encoded = pd.get_dummies(df_clean, columns=cat_cols_clean, drop_first=False)
print(f"\nVariables después de encoding: {df_encoded.shape[1]} columnas")

# ── Separar X / y ─────────────────────────────────────────────────────
X = df_encoded.drop(columns=[TARGET])
y = df_encoded[TARGET]
print(f"X: {X.shape}  |  y: {y.shape}")

### 4.7 División 70 / 15 / 15 y normalización

In [ ]:
# 1️  Separar test (15%)
X_temp, X_test, y_temp, y_test = train_test_split(
    X, y, test_size=0.15, random_state=RANDOM_STATE
)

# 2️  Separar validación del resto → val_ratio sobre lo que queda
val_ratio = 0.15 / (1 - 0.15)
X_train, X_val, y_train, y_val = train_test_split(
    X_temp, y_temp, test_size=val_ratio, random_state=RANDOM_STATE
)

# 3️  Normalización — fit SOLO en train
scaler = StandardScaler()
X_train_sc = scaler.fit_transform(X_train)
X_val_sc   = scaler.transform(X_val)
X_test_sc  = scaler.transform(X_test)

n_total = len(X)
print("División del dataset:")
print(f"  Train      : {len(X_train):>5} muestras ({len(X_train)/n_total*100:.1f}%)")
print(f"  Validación : {len(X_val):>5} muestras ({len(X_val)/n_total*100:.1f}%)")
print(f"  Test       : {len(X_test):>5} muestras ({len(X_test)/n_total*100:.1f}%)")
print("\n Normalización aplicada (StandardScaler fit sobre train únicamente)")

---
##  Resultados del proyecto — Comparación de modelos
*(Valores obtenidos de `results/metrics.csv`)*

In [ ]:
import os

metrics_path = "../results/metrics.csv"
metrics_df   = pd.read_csv(metrics_path)

print(" Tabla comparativa de métricas:")
display(metrics_df)

In [ ]:
reg = metrics_df[metrics_df["Tipo"] == "Regresión"].copy()

COLORS = {
    "SVR":                 "#2A9D8F",
    "Árbol de Decisión":   "#E9C46A",
    "Random Forest":       "#264653",
    "Red Neuronal (MLP)":  "#A8DADC",
    "Red Neuronal":        "#A8DADC",
}
bar_colors = [COLORS.get(m, "#888") for m in reg["Modelo"]]

fig, axes = plt.subplots(1, 3, figsize=(15, 5))
fig.suptitle("Comparación de Modelos de Regresión", fontsize=14, fontweight="bold")

for ax, (col, label, note) in zip(axes, [
    ("MAE",  "MAE (USD)",  "↓ menor = mejor"),
    ("RMSE", "RMSE (USD)", "↓ menor = mejor"),
    ("R²",   "R²",         "↑ mayor = mejor"),
]):
    bars = ax.bar(reg["Modelo"], reg[col], color=bar_colors,
                  edgecolor="white", width=0.55)
    ax.bar_label(bars, fmt="%.2f", padding=3, fontsize=8)
    ax.set_title(f"{col}  ({note})", fontsize=11)
    ax.set_xticks(range(len(reg)))
    ax.set_xticklabels(reg["Modelo"], rotation=28, ha="right", fontsize=9)
    ax.set_ylabel(label)
    ax.grid(axis="y", alpha=0.3)
    ax.spines[["top", "right"]].set_visible(False)

plt.tight_layout()
plt.savefig(f"{GRAPHS_PATH}comparacion_modelos_nb.png", dpi=150, bbox_inches="tight")
plt.show()
print(" Gráfica guardada")

In [ ]:
reg_sorted = reg.sort_values("R²", ascending=True)
colors_sorted = [COLORS.get(m, "#888") for m in reg_sorted["Modelo"]]

fig, ax = plt.subplots(figsize=(9, 4))
bars = ax.barh(reg_sorted["Modelo"], reg_sorted["R²"],
               color=colors_sorted, edgecolor="white", height=0.5)
ax.bar_label(bars, fmt="%.4f", padding=5, fontsize=10)
ax.set_xlabel("R² (coeficiente de determinación)", fontsize=11)
ax.set_title("Ranking R² — Modelos de Regresión", fontsize=13, fontweight="bold")
ax.set_xlim(0, 1.08)
ax.axvline(1, color="gray", linestyle="--", linewidth=1, alpha=0.5)
ax.grid(axis="x", alpha=0.3)
ax.spines[["top", "right"]].set_visible(False)
plt.tight_layout()
plt.show()

best = reg_sorted.iloc[-1]
print(f"\n Mejor modelo: {best['Modelo']}")
print(f"   R² = {best['R²']}  |  MAE = {best['MAE']} USD  |  RMSE = {best['RMSE']} USD")

---
##  Resumen del ciclo de vida ML

In [ ]:
summary = pd.DataFrame({
    "Paso": range(1, 7),
    "Etapa": [
        "Recopilación de datos",
        "Elección de medida de éxito (MAE, RMSE, R², Accuracy, F1)",
        "Protocolo de evaluación (70/15/15, random_state=42)",
        "Preparación de datos (limpieza, encoding, normalización)",
        "Punto de referencia (Regresión Logística + SVR)",
        "Modelos avanzados y comparación (Árbol, RF, MLP)"
    ],
    "Estado": ["ok!"] * 6
})

display(summary)
print("\n Ciclo de vida ML completado correctamente.")